In [0]:
%python
from pyspark.sql import functions as F
from pyspark.sql.window import Window

BRONZE_PATH = "/mnt/usvikformula1dl/bronze"
SILVER_PATH = "/mnt/usvikformula1dl/silver"

customer_bronze_path = f"{BRONZE_PATH}/customer_master"
customer_silver_path = f"{SILVER_PATH}/customer_master"

customer_master_df =spark.read.format("delta").load(customer_bronze_path)
window_spec= Window.partitionBy("customer_id").orderBy(F.desc("update_ts"))
final_cus_master_df=customer_master_df.withColumn("row_num",F.row_number().over(window_spec))
final_cus_master_df_1=final_cus_master_df.filter(F.col('row_num')==1).drop('row_num')
final_cus_master_df_1.write.mode("overwrite").format("delta").save(customer_silver_path)
#display(final_cus_master_df_1)

#validation
print("Total Silver records:", final_cus_master_df_1.count())

print(
    "Unique customer IDs:",
    final_cus_master_df_1
        .select("customer_id")
        .distinct()
        .count()
)